# BoltzDesign1: RBX1 RING Core Binder Design

**Target**: RBX1 RING core, residues 32–83 (52 AA, chain B)  
**Why truncated**: Full 108 AA RBX1 contains disordered N-term (1–31) and C-term (84–108) tails that attract false binders. This uses only the structured zinc-binding RING core.

**Pipeline**:
1. BoltzDesign1 — jointly optimises backbone + sequence via gradient descent through Boltz-1, conditioned on RING hotspots
2. Boltz-2 — rescore all designed sequences as complexes, filter by ipTM ≥ 0.70

BoltzDesign1 outputs sequences directly — no separate redesign step needed.

**Hotspot residues** (1-based in the 52 AA fragment):  
`4, 6, 12, 14, 15, 23, 24, 26, 28, 52`  
These correspond to the RING H2 loop and zinc-coordinating surface (original positions 35, 37, 43, 45, 46, 54, 55, 57, 59, 83).

In [ ]:
#@title 🛠️ Setup (~3 minutes) — kernel will restart once
%%time
import os

if not os.path.isdir('BoltzDesign1'):
    !git clone -q https://github.com/yehlincho/BoltzDesign1.git
    !cd /content/BoltzDesign1/boltz && pip install -q git+https://github.com/prody/ProDy.git .
    !pip install -q pypdb py3Dmol logmd
    !cd /content/BoltzDesign1/LigandMPNN && bash get_model_params.sh './model_params'
    exit()  # kernel restarts — re-run all cells after restart

print('Setup complete — ready to load models')

In [ ]:
#@title 🛠️ Load BoltzDesign1 model
from boltz.main import download
from pathlib import Path
download(Path('.'))

import os, sys, shutil, warnings
warnings.simplefilter('ignore')
sys.path.append('./BoltzDesign1/boltzdesign')

from boltzdesign_utils import *
from input_utils import *
from utils import *
import pandas as pd, yaml

ccd_path = 'ccd.pkl'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

predict_args = {
    'recycling_steps': 0,
    'sampling_steps': 200,
    'diffusion_samples': 1,
    'write_confidence_summary': True,
    'write_full_pae': False,
    'write_full_pde': False,
}
boltz_model = get_boltz_model('boltz1_conf.ckpt', predict_args, device)
boltz_model.train()
print(f'BoltzDesign1 model loaded on {device}')

In [ ]:
#@title 📂 Upload target PDB
# Upload rbx1_ring_core.pdb  (residues 32-83, chain B, 52 AA)
from google.colab import files
print('Upload rbx1_ring_core.pdb')
uploaded = files.upload()
TARGET_PDB = list(uploaded.keys())[0]

# Verify
residues, chains = set(), set()
with open(TARGET_PDB) as f:
    for line in f:
        if line.startswith('ATOM'):
            residues.add(int(line[22:26]))
            chains.add(line[21])
print(f'Chains: {sorted(chains)}  |  Residues: {min(residues)}-{max(residues)} (n={len(residues)})')
assert len(residues) == 52, f'Expected 52 residues, got {len(residues)}'
print('✓ Correct — 52 AA structured RING core')

In [ ]:
#@title 📄 Generate input YAML
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

TARGET_NAME      = 'rbx1_ring_core'
TARGET_CHAIN     = 'B'
BINDER_CHAIN     = 'A'
# Hotspot residues — 1-based in the 52 AA fragment
# RING H2 loop + zinc-coordinating surface
CONTACT_RESIDUES = '4,6,12,14,15,23,24,26,28,52'

config_obj = Config(main_dir=f'/content/inputs/{TARGET_NAME}_binder')
config_obj.setup_directories()

yaml_dict, yaml_dir = generate_yaml_config(
    input_type='pdb',
    pdb_path=TARGET_PDB,
    target_type='protein',
    target_name=TARGET_NAME,
    pdb_target_ids=TARGET_CHAIN,
    target_mols='',
    custom_target_input='',
    custom_target_ids='',
    binder_id=BINDER_CHAIN,
    use_msa=False,
    contact_residues=CONTACT_RESIDUES,
    modifications='',
    modifications_positions='',
    modification_target='',
    constraint_target=TARGET_CHAIN,
    config_obj=config_obj
)

config_path = '/content/BoltzDesign1/boltzdesign/configs/default_ppi_config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

print(f'YAML saved: {yaml_dir}')
print(f'Pocket conditioning ON — hotspots: {CONTACT_RESIDUES}')

In [ ]:
#@title 🔧 Define generate_yaml_config (missing from imported modules)
import logging
from pathlib import Path

logger = logging.getLogger(__name__)

def generate_yaml_config(
    input_type, pdb_path, target_type, target_name, pdb_target_ids,
    target_mols, custom_target_input, custom_target_ids, binder_id,
    use_msa, contact_residues, modifications, modifications_positions,
    modification_target, constraint_target, config_obj
):
    constraints = None
    modifications_dict = None
    if contact_residues or modifications:
        target_ids = pdb_target_ids if input_type == "pdb" else custom_target_ids
        target_ids_list = [str(x.strip()) for x in target_ids.split(",")] if target_ids else []
        target_id_map = {id: c for id, c in zip(target_ids_list, 'BCDEFGHIJKLMNOPQRSTUVWXYZ')}
        constraints, modifications_dict = process_design_constraints(
            target_id_map, modifications, modifications_positions,
            modification_target, contact_residues, constraint_target, binder_id
        )

    target = []
    if input_type == "pdb":
        if pdb_path is not None:
            if not Path(pdb_path).is_file():
                raise FileNotFoundError(f"Could not find local PDB: {pdb_path}")
        else:
            download_pdb(target_name, config_obj.PDB_DIR)
            pdb_path = config_obj.PDB_DIR / f"{target_name}.pdb"

        if target_type == 'protein':
            sequences = get_chains_sequence(pdb_path)
            target = [sequences[id] for id in pdb_target_ids.split(",")]
        else:
            raise ValueError(f"Unsupported target type: {target_type}")
    else:
        target = custom_target_input.split(",") if custom_target_input else [target_name]

    return generate_yaml_for_target_binder(
        target_name, target_type, target,
        config=config_obj, binder_id=binder_id,
        constraints=constraints,
        modifications=modifications_dict['data'] if modifications_dict else None,
        modification_target=modifications_dict['target'] if modifications_dict else None,
        use_msa=use_msa
    )

print("generate_yaml_config defined")


In [ ]:
#@title 🧬 Design settings

DESIGN_SAMPLES = 50   #@param {type:"integer"}  — number of sequences to generate
LENGTH_MIN     = 65   #@param {type:"integer"}
LENGTH_MAX     = 95   #@param {type:"integer"}
OPTIMIZER      = 'SGD' #@param ["SGD", "AdamW"]

config['binder_chain']                    = BINDER_CHAIN
config['non_protein_target']              = False
config['msa_max_seqs']                    = 4096
config['length_min']                      = LENGTH_MIN
config['length_max']                      = LENGTH_MAX
config['optimizer_type']                  = OPTIMIZER
config['pocket_conditioning']             = True
config['optimize_contact_per_binder_pos'] = True   # correct for PPI
config['mask_ligand']                     = False   # no ligand masking for protein target

# Iteration schedule
config['pre_iteration']     = 30
config['soft_iteration']    = 80
config['temp_iteration']    = 45
config['hard_iteration']    = 5
config['semi_greedy_steps'] = 2

# Loss weights (PPI)
loss_scales = {
    'con_loss':   1.0,   # intra-binder contacts
    'i_con_loss': 1.0,   # binder–RBX1 interface contacts
    'plddt_loss': 0.1,
    'pae_loss':   0.4,
    'i_pae_loss': 0.1,
    'rg_loss':    0.0,
}

print(f'Designs:        {DESIGN_SAMPLES}')
print(f'Binder length:  {LENGTH_MIN}–{LENGTH_MAX} AA')
print(f'Target:         RBX1 RING core 32–83, chain B (52 AA)')
print(f'Hotspots:       {CONTACT_RESIDUES}')

In [ ]:
#@title 🚀 Run BoltzDesign1
# Each design run jointly optimises backbone + sequence.
# Output: CIF files containing the designed structure and sequence.

MAIN_DIR     = 'outputs'
VERSION_NAME = f'protein_{TARGET_NAME}'
os.makedirs(MAIN_DIR, exist_ok=True)

run_boltz_design(
    boltz_path=shutil.which('boltz'),
    main_dir=MAIN_DIR,
    yaml_dir=os.path.dirname(yaml_dir),
    boltz_model=boltz_model,
    ccd_path=ccd_path,
    design_samples=DESIGN_SAMPLES,
    version_name=VERSION_NAME,
    config=config,
    loss_scales=loss_scales,
    num_workers=0,
    show_animation=True,
    save_trajectory=False,
    redo_boltz_predict=False,
)
print('\nBoltzDesign1 complete')

In [ ]:
#@title 📊 Extract designed sequences from BoltzDesign1 output
import glob, re

results_dir = os.path.join(MAIN_DIR, VERSION_NAME, 'results_final')
results_csv = os.path.join(results_dir, 'rmsd_results.csv')

df = pd.read_csv(results_csv)
print(f'Generated {len(df)} designs')
print(df[['target','length','iptm','complex_plddt','apo_holo_rmsd']]
      .sort_values('iptm', ascending=False)
      .to_string())

# Extract binder sequence (chain A) from each output CIF
def extract_chain_seq_from_cif(cif_path, chain='A'):
    AA3 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
           'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
           'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'}
    cols = {}; seq = {}; in_loop = False
    with open(cif_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('_atom_site.'):
                cols[line.split('.')[-1].strip()] = len(cols)
                in_loop = True; continue
            if not in_loop or not line.startswith('ATOM'): continue
            parts = line.split()
            try:
                if parts[cols['label_asym_id']] != chain: continue
                if parts[cols['label_atom_id']] != 'CA': continue
                rn  = int(parts[cols['label_seq_id']])
                res = parts[cols['label_comp_id']]
                seq[rn] = AA3.get(res, 'X')
            except: continue
    return ''.join(seq[k] for k in sorted(seq))

designs = []
for _, row in df.iterrows():
    cif_files = sorted(glob.glob(f"{results_dir}/*.cif"))
    # match by target name / iteration index
    for cif in cif_files:
        seq = extract_chain_seq_from_cif(cif, chain='A')
        if seq and len(seq) == int(row.get('length', 0)):
            name = f"BZD_{TARGET_NAME}_len{len(seq)}_i{int(row.get('iteration',0))}"
            designs.append({
                'name': name, 'seq': seq, 'length': len(seq),
                'bd_iptm': row.get('iptm',''),
                'bd_plddt': row.get('complex_plddt',''),
                'cif': cif,
            })
            break

# Deduplicate by sequence
seen = set()
unique = []
for d in designs:
    if d['seq'] not in seen:
        seen.add(d['seq']); unique.append(d)

# Give clean sequential names
for i, d in enumerate(unique):
    d['name'] = f'BZD_{i+1:03d}_len{d["length"]}'

print(f'\nUnique sequences: {len(unique)}')
print(f'Length range: {min(d["length"] for d in unique)}–{max(d["length"] for d in unique)} AA')
for d in unique[:5]:
    print(f"  {d['name']:<30} {d['length']} AA  bd_ipTM={d['bd_iptm']:.3f}")

In [ ]:
#@title 🔬 ProteinMPNN — 9 sequences per BoltzDesign1 backbone (3 temps × 3 seqs)
# Takes the 15 BoltzDesign1 output CIFs and samples diverse sequences from each backbone.
# Design chain A (binder), fix chain B (RBX1). Total: 15 × 9 = 135 sequences.

import sys, copy, torch, numpy as np, glob, os

if not os.path.isdir('ProteinMPNN'):
    !git clone -q https://github.com/dauparas/ProteinMPNN.git

sys.path.append('/content/ProteinMPNN')
from protein_mpnn_utils import (
    _S_to_seq, tied_featurize, parse_PDB, StructureDatasetPDB, ProteinMPNN
)

# Load model
mpnn_device   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
ckpt_path     = '/content/ProteinMPNN/vanilla_model_weights/v_48_020.pt'
ckpt          = torch.load(ckpt_path, map_location=mpnn_device)
mpnn_model    = ProteinMPNN(
    num_letters=21, node_features=128, edge_features=128, hidden_dim=128,
    num_encoder_layers=3, num_decoder_layers=3, augment_eps=0.0,
    k_neighbors=ckpt['num_edges']
)
mpnn_model.to(mpnn_device)
mpnn_model.load_state_dict(ckpt['model_state_dict'])
mpnn_model.eval()
print(f'ProteinMPNN loaded on {mpnn_device}')

# Convert BoltzDesign1 CIFs → PDB (both chains)
AA3 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
       'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
       'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'}

pdb_dir = os.path.join(MAIN_DIR, VERSION_NAME, 'pdb')
os.makedirs(pdb_dir, exist_ok=True)

def cif_to_pdb(cif_path, pdb_path):
    cols = {}; lines_out = []; in_loop = False; serial = 1
    with open(cif_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('_atom_site.'):
                cols[line.split('.')[-1].strip()] = len(cols); in_loop = True; continue
            if not in_loop or not line.startswith('ATOM'): continue
            parts = line.split()
            try:
                atom    = parts[cols['label_atom_id']]
                chain   = parts[cols['label_asym_id']]
                resnum  = int(parts[cols['label_seq_id']])
                resname = parts[cols['label_comp_id']]
                x,y,z   = float(parts[cols['Cartn_x']]),float(parts[cols['Cartn_y']]),float(parts[cols['Cartn_z']])
            except: continue
            if resname not in AA3: continue
            lines_out.append(f'ATOM  {serial:5d}  {atom:<3s} {resname} {chain}{resnum:4d}    {x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           {atom[0]}\n')
            serial += 1
    with open(pdb_path, 'w') as f:
        f.writelines(lines_out); f.write('END\n')

cif_files = sorted(glob.glob(f'{MAIN_DIR}/{VERSION_NAME}/results_final/*.cif'))
pdb_entries = []
for cif in cif_files:
    name = os.path.basename(cif).replace('.cif','')
    pdb  = f'{pdb_dir}/{name}.pdb'
    cif_to_pdb(cif, pdb)
    pdb_entries.append({'name': name, 'cif': cif, 'pdb': pdb})
print(f'Converted {len(pdb_entries)} CIFs → PDB')

# Run ProteinMPNN
MPNN_TEMPS    = [0.05, 0.10, 0.15]
SEQS_PER_TEMP = 3
alphabet      = 'ACDEFGHIKLMNPQRSTVWYX'
omit_AAs_np   = np.array([aa in 'X' for aa in alphabet], dtype=np.float32)
bias_AAs_np   = np.zeros(len(alphabet))

mpnn_seqs = []
for entry in pdb_entries:
    try:
        pdb_dict_list = parse_PDB(entry['pdb'], input_chain_list=['A','B'])
    except Exception as e:
        print(f"  SKIP {entry['name']}: {e}"); continue
    if not pdb_dict_list: continue

    dataset      = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=20000)
    chain_id_dict = {pdb_dict_list[0]['name']: (['A'], ['B'])}
    binder_len    = len(pdb_dict_list[0].get('seq_chain_A', ''))

    for temp in MPNN_TEMPS:
        with torch.no_grad():
            for protein in dataset:
                batch = [copy.deepcopy(protein) for _ in range(SEQS_PER_TEMP)]
                try:
                    X, S, mask, lengths, chain_M, chain_encoding_all, \
                    chain_list_list, visible_list_list, masked_list_list, \
                    masked_chain_length_list_list, chain_M_pos, omit_AA_mask, \
                    residue_idx, dihedral_mask, tied_pos_list_of_lists_list, \
                    pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, \
                    tied_beta = tied_featurize(batch, mpnn_device, chain_id_dict, None, None, None, None, None)
                except Exception as e:
                    print(f"  SKIP {entry['name']} T={temp}: {e}"); continue

                sample_dict = mpnn_model.sample(
                    X, torch.randn(chain_M.shape, device=mpnn_device),
                    S, chain_M, chain_encoding_all, residue_idx,
                    mask=mask, temperature=temp,
                    omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np,
                    chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask,
                    pssm_coef=pssm_coef, pssm_bias=pssm_bias,
                    pssm_multi=0.0, pssm_log_odds_flag=False,
                    pssm_log_odds_mask=(pssm_log_odds_all > 0.0).float(),
                    pssm_bias_flag=False, bias_by_res=bias_by_res_all
                )
                for b_ix in range(SEQS_PER_TEMP):
                    full_seq = _S_to_seq(sample_dict['S'][b_ix], chain_M[b_ix])
                    seq = full_seq[:binder_len]
                    mpnn_seqs.append({
                        'name':     f"MPNN_{entry['name']}_t{temp:.2f}_s{b_ix+1}",
                        'seq':      seq,
                        'length':   len(seq),
                        'backbone': entry['name'],
                        'temp':     temp,
                        'source':   'ProteinMPNN',
                        'bd_iptm':  '',
                        'bd_plddt': '',
                    })

print(f'\nProteinMPNN generated {len(mpnn_seqs)} sequences ({len(pdb_entries)} backbones × {len(MPNN_TEMPS)} temps × {SEQS_PER_TEMP} seqs)')


In [ ]:
#@title 💾 Write FASTA + Boltz-2 YAMLs

# Truncated RBX1 — structured RING core only, no disordered tails
RBX1_RING_CORE = 'LWAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHC'  # res 32-83, 52 AA

OUT_DIR  = 'bzd_outputs'
YAML_DIR = f'{OUT_DIR}/boltz_yamls'
os.makedirs(YAML_DIR, exist_ok=True)

# Sort best-first by BoltzDesign1 ipTM
unique.sort(key=lambda d: -(float(d['bd_iptm']) if d['bd_iptm'] else 0))

fasta_path = f'{OUT_DIR}/bzd_candidates.fasta'
with open(fasta_path, 'w') as f:
    for d in unique:
        f.write(f">{d['name']} bd_iptm={d['bd_iptm']}\n{d['seq']}\n")
print(f'FASTA: {fasta_path}  ({len(unique)} sequences)')

yaml_tmpl = 'version: 1\nsequences:\n  - protein:\n      id: A\n      sequence: {binder}\n  - protein:\n      id: B\n      sequence: {rbx1}\n'
for d in unique:
    with open(f"{YAML_DIR}/{d['name']}_complex.yaml", 'w') as f:
        f.write(yaml_tmpl.format(binder=d['seq'], rbx1=RBX1_RING_CORE))
print(f'Boltz-2 YAMLs: {YAML_DIR}/  ({len(unique)} files)')
print(f'RBX1 target:   {RBX1_RING_CORE}  ({len(RBX1_RING_CORE)} AA)')

In [ ]:
#@title 🔬 Rescore with Boltz-2
try:
    import boltz as _b
    print('Boltz available')
except ImportError:
    !pip install -q boltz

BOLTZ_OUT = f'{OUT_DIR}/boltz_results'
os.makedirs(BOLTZ_OUT, exist_ok=True)

yaml_files = sorted(glob.glob(f'{YAML_DIR}/*.yaml'))
print(f'Rescoring {len(yaml_files)} sequences...')

for yf in yaml_files:
    name = os.path.basename(yf).replace('_complex.yaml', '')
    out  = f'{BOLTZ_OUT}/{name}'
    if glob.glob(f'{out}/**/confidence_*.json', recursive=True):
        continue  # already done
    !boltz predict {yf} --out_dir {out} --accelerator gpu \
        --no_kernels \
        --recycling_steps 3 --sampling_steps 200 --diffusion_samples 1 \

print('Rescoring complete')

In [ ]:
#@title 🏆 Rank and filter results
import json

scored = []
for d in unique:
    conf_files = glob.glob(f"{BOLTZ_OUT}/{d['name']}/**/confidence_*.json", recursive=True)
    if not conf_files: continue
    with open(conf_files[0]) as f:
        conf = json.load(f)
    scored.append({
        **d,
        'iptm':   conf.get('iptm', 0),
        'iplddt': conf.get('interface_plddt', conf.get('complex_plddt', 0)),
        'ptm':    conf.get('ptm', 0),
    })

scored.sort(key=lambda r: -r['iptm'])
passing    = [r for r in scored if r['iptm'] >= 0.70]
borderline = [r for r in scored if 0.65 <= r['iptm'] < 0.70]

print(f'Scored:         {len(scored)}')
print(f'PASS  (≥0.70):  {len(passing)}')
print(f'BORDER (≥0.65): {len(borderline)}')
print(f'FAIL  (<0.65):  {len(scored)-len(passing)-len(borderline)}')

print(f'\nTop 20 by Boltz-2 ipTM:')
print(f'{"name":<30} {"ipTM":>6} {"ipLDDT":>7} {"pTM":>5} {"bd_ipTM":>8} {"len":>5}')
print('-'*65)
for r in scored[:20]:
    flag = ' PASS' if r['iptm'] >= 0.70 else (' BORDER' if r['iptm'] >= 0.65 else '')
    print(f"{r['name']:<30} {r['iptm']:>6.3f} {r['iplddt']:>7.3f} {r['ptm']:>5.3f} "
          f"{float(r['bd_iptm']) if r['bd_iptm'] else 0:>8.3f} {r['length']:>5}{flag}")

In [ ]:
#@title 📦 Download results
import csv as _csv
from google.colab import files

PASS_FASTA = f'{OUT_DIR}/bzd_passing.fasta'
PASS_CSV   = f'{OUT_DIR}/bzd_passing.csv'

keep = passing + borderline
with open(PASS_FASTA, 'w') as f:
    for r in keep:
        f.write(f">{r['name']} iptm={r['iptm']:.3f} bd_iptm={r['bd_iptm']}\n{r['seq']}\n")

with open(PASS_CSV, 'w', newline='') as f:
    fields = ['name','length','iptm','iplddt','ptm','bd_iptm','bd_plddt','seq']
    w = _csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
    w.writeheader(); w.writerows(keep)

print(f'Saved {len(keep)} sequences (PASS + BORDERLINE)')
files.download(PASS_FASTA)
files.download(PASS_CSV)